# RC Model Validation: Python vs. MATLAB

**Objective:** Validate the Python RC model implementation against MATLAB reference.

**Status:** Model corrections applied:
1. ✓ Hour-of-day indexing for schedules: `(hour_counter - 1) % 24`
2. ✓ Config parameters: infiltration/ventilation rates corrected
3. ✓ Equipment schedule: synced with MATLAB

**Expected Results:** Heating ≈ 54 MWh, Cooling ≈ -0.3 MWh


In [3]:
import os, sys
import numpy as np
import pandas as pd
from scipy.io import loadmat
from pathlib import Path

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from core.bootstrap import create_facade

PROJECT_ID = "rc-model-validation"
VARIANT_ID = "Val"
print("✓ Setup complete")

✓ Setup complete


In [ ]:
# Validate
df_diff = df_res_py.subtract(df_res_mat, fill_value=0)
df_diff_ida = df_res_py[df_res_ida.columns].subtract(df_res_ida, fill_value=0)

print("\n" + "="*90)
print("VALIDATION: Python vs. MATLAB vs. IDA ICE")
print("="*90)

# Temperature
temp_diff = df_diff["temperature_air_room"]
temp_max = np.max(np.abs(temp_diff))
temp_rmse = np.sqrt(np.mean(temp_diff**2))
print(f"\nTEMPERATURE (°C)")
print(f"  Max Error:  {temp_max:>8.4f}°C  |  RMSE: {temp_rmse:>8.4f}°C  |  Mean: {np.mean(temp_diff):>+8.4f}°C")

# Heating
heat_diff = df_diff["output_heating_power"]
py_heat = df_res_py['output_heating_power'].sum()/1e6
ml_heat = df_res_mat['output_heating_power'].sum()/1e6
heat_pct = ((py_heat/ml_heat-1)*100) if ml_heat != 0 else 0
print(f"\nHEATING (MWh)")
print(f"  Python: {py_heat:>8.2f}  |  MATLAB: {ml_heat:>8.2f}  |  Diff: {py_heat-ml_heat:>+8.2f} ({heat_pct:>+6.1f}%)")

# Cooling
cool_diff = df_diff["output_cooling_power"]
py_cool = df_res_py['output_cooling_power'].sum()/1e6
ml_cool = df_res_mat['output_cooling_power'].sum()/1e6
print(f"\nCOOLING (MWh)")
print(f"  Python: {py_cool:>8.2f}  |  MATLAB: {ml_cool:>8.2f}  |  Diff: {py_cool-ml_cool:>+8.2f}")

# IDA comparison
temp_diff_ida = df_diff_ida["temperature_air_room"]
temp_rmse_ida = np.sqrt(np.mean(temp_diff_ida**2))
print(f"\nPython vs. IDA ICE")
print(f"  Temp RMSE: {temp_rmse_ida:>8.4f}°C")

# Summary
print("\n" + "="*90)
if temp_max < 0.5 and abs(heat_pct) < 1.0:
    print("✓✓✓ VALIDATION SUCCESSFUL")
elif temp_max < 1.0 and abs(heat_pct) < 5.0:
    print("✓ VALIDATION ACCEPTABLE")
else:
    print("⚠ Discrepancies remain - further investigation needed")
print("="*90)

In [ ]:
# Run simulation and load all data
facade_Val = create_facade(PROJECT_ID, VARIANT_ID)
facade_Val.run_simulation(PROJECT_ID, VARIANT_ID)
df_res_py = facade_Val._result.load_raw().drop(columns=["temperature_outdoor_air"], axis=1)

# Load MATLAB reference
mat_path = Path(PROJECT_ROOT) / 'projects' / 'rc-model-validation' / 'mat-reference' / 'matlab_ref_results.mat'
mat_data = loadmat(mat_path, squeeze_me=True)
df_res_mat = pd.DataFrame({
    'output_heating_power': mat_data['output_heating_power'],
    'output_cooling_power': mat_data['output_cooling_power'],
    'output_lighting_electricity': mat_data['output_lighting_electricity'],
    'output_equipment_electricity': mat_data['output_equipment_electricity']
})
temp_mat_array = mat_data['output_temperatures']
row_names_all = df_res_py.columns.tolist()
row_names = row_names_all[:-4]
df_temp_outputs = pd.DataFrame(data=temp_mat_array, columns=row_names)
df_res_mat = pd.concat([df_temp_outputs, df_res_mat], axis=1)
df_res_mat.index = df_res_py.index

# Load IDA ICE reference
ida_path = Path(PROJECT_ROOT) / 'projects' / 'rc-model-validation' / 'mat-reference' / 'ida_results_ver2.mat'
ida_data = loadmat(ida_path, squeeze_me=True)
df_res_ida = pd.DataFrame({
    "temperature_air_room": ida_data['ida_air_temp'],
    "output_heating_power": ida_data['ida_heating_power'],
    "output_cooling_power": ida_data['ida_cooling_power'],
})
df_res_ida.index = df_res_py.index
print("✓ Simulation complete. All data loaded.")